# 01 · Data and validation

**Question:** What does the validation design actually test?

This notebook renders recorded **competition-data** evidence. It verifies the committed artifact hashes, not private out-of-fold predictions. No credentials, raw comments, weight downloads, or training are needed. Full private metric recomputation remains `uv run jigsaw review`.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Competition data | 2,029 training rows | recorded local cross-validation")
print("Verification: aggregate file checksums and provenance; no model fitting.")

names = {"comment_only": "Comment-only TF-IDF", "rule_examples": "Rule/example TF-IDF",
         "semantic_margin": "Frozen semantic margin", "semantic_classifier": "Semantic classifier"}
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records):
    return pd.DataFrame([{"Model": names[r["model"]], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]} for r in records]).round(4)


Competition data | 2,029 training rows | recorded local cross-validation
Verification: aggregate file checksums and provenance; no model fitting.


## Two observed rules, not broad policy coverage
The recorded training audit exposes different violation prevalence across the two rules. Count and rate are shown together so sample size stays visible.

In [2]:
audit = baseline["audit"]
by_rule = pd.DataFrame(audit["by_rule"]).rename(columns={"rule": "Rule", "size": "Rows", "mean": "Violation rate"})
by_rule["Rule"] = by_rule["Rule"].str.split(":").str[0]
display(by_rule.round(4))
display(pd.DataFrame({"Audit finding": ["Duplicate training bodies", "Train / preview-test overlap"],
                      "Count": [audit["duplicate_training_bodies"], audit["train_test_body_overlap"]]}))

,Rule,Rows,Violation rate
0,No Advertising,1012,0.4328
1,No legal advice,1017,0.5831


,Audit finding,Count
0,Duplicate training bodies,162
1,Train / preview-test overlap,10


## Leakage controls
**Familiar-rule validation** stratifies by rule and target while grouping normalized duplicate comments. **Held-out-rule validation** excludes all training rows from the evaluated rule. Both purge training rows whose body or supplied examples contain a validation body. Vocabulary and learned similarity classifiers are fitted only on retained training-fold rows.

Provided validation examples remain legitimate inputs; their labels are not added as training observations. The semantic experiment reused the original saved splits. This aggregate-only view does not re-run the purging audit or reconstruct row assignments.

In [3]:
config = baseline["provenance"]["config"]
display(pd.DataFrame({"Predeclared setting": ["Seed", "Familiar-rule folds", "Observed labeled rules"],
                      "Value": [config["seed"], config["folds"], len(audit["train_rules"])]}))
print("Protocol limitations: exact-text isolation does not prove near-duplicate or shared-origin isolation.")

,Predeclared setting,Value
0,Seed,2025
1,Familiar-rule folds,3
2,Observed labeled rules,2


Protocol limitations: exact-text isolation does not prove near-duplicate or shared-origin isolation.


## Metric contract
The official overview names **column-averaged AUC**. The project reports **rule macro ROC AUC**, the equal-weight mean of the rule-specific AUCs, and pooled AUC separately. The official overview does not expose executable scoring code.

Log loss, Brier score, average precision, and calibration error diagnose different properties; AUC alone does not establish probability calibration. Threshold metrics use a predeclared 0.5 threshold. Only two labeled rules means two transfer cases, not a representative sample of future policies.

Raw comment-length distributions and row-level errors are not inferred from these aggregate files. Continue to [02 · Baseline](02_baseline_and_review.ipynb).